CTR by Age Group
===
Difficulty: Hard

Problem Description:
===================
Given `search_events` and `users` tables, find the **top 3 age groups** (bucketed by decade:
20s → group 2, 30s → group 3, etc.) with the highest **click-through rate (CTR)** in 2021.
CTR = clicks / total searches. If two groups tie, the older group ranks higher.

Sample Input:
```
users:         search_events:
| user_id | age |    | event_id | user_id | action | year |
|---------|-----|    |----------|---------|--------|------|
| 1       | 25  |    | 1        | 1       | search | 2021 |
| 2       | 32  |    | 2        | 1       | click  | 2021 |
| 3       | 45  |    | 3        | 2       | search | 2021 |
```

Sample Output:
```
| age_group | ctr  |
|-----------|------|
| 4         | 0.75 |
| 2         | 0.50 |
| 3         | 0.33 |
```

In [1]:
import pandas as pd

users = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'age':     [25, 32, 25, 45, 32, 45, 25, 18]
})

search_events = pd.DataFrame({
    'event_id': range(1, 21),
    'user_id':  [1,1,1,1, 2,2,2, 3,3, 4,4,4,4, 5,5, 6, 7,7,7, 8],
    'action':   [
        'search','click','search','search',   # User 1 (25): 1 click / 3 search
        'search','search','click',            # User 2 (32): 1 click / 2 search
        'search','click',                     # User 3 (25): 1 click / 1 search
        'search','click','click','search',    # User 4 (45): 2 click / 2 search
        'click','search',                     # User 5 (32): 1 click / 1 search
        'search',                             # User 6 (45): 0 click / 1 search
        'search','click','search',            # User 7 (25): 1 click / 2 search
        'search'                              # User 8 (18): 0 click / 1 search
    ],
    'year': [2021]*20
})
print(users)
print(search_events.head())

   user_id  age
0        1   25
1        2   32
2        3   25
3        4   45
4        5   32
5        6   45
6        7   25
7        8   18
   event_id  user_id  action  year
0         1        1  search  2021
1         2        1   click  2021
2         3        1  search  2021
3         4        1  search  2021
4         5        2  search  2021


In [21]:
users
bins = [0,21,31,41,51]
labels = ['1','2','3','4']

users['age_bucket'] = pd.cut(users['age'],bins=bins,labels=labels)

merged = pd.merge(left=users,right=search_events,how='left')

# merged = merged[merged['action'] == 'search'].groupby('age_bucket').count().reset_index()

merged = merged.groupby('age_bucket')['action'].agg(
                                    clicks = lambda x:(x=='click').sum()
                                    ,total = 'count'                                    )

merged['CTR'] = (merged['clicks']/merged['total']).round(2)

# merged.reset_index()

merged

,clicks,total,CTR
age_bucket,,,
1,0,1,0.00
2,3,9,0.33
3,2,5,0.40
4,2,5,0.40


**Concepts to use:**
1. **`merge()`** — join events with users on `user_id` to attach age.
2. **`age // 10`** — integer division buckets age into decades (25 → 2, 32 → 3, 45 → 4).
3. **Conditional aggregation** — `(df['action'] == 'click').sum()` within groupby using `apply` or `assign`.
4. **`sort_values([ctr, age_group], ascending=[False, False])`** — tiebreak by older group.

In [16]:
# Optimised Solution
def ctr_by_age(events, users):
    # Step 1: merge
    df = events.merge(users, on='user_id')

    # Step 2: filter to 2021
    df = df[df['year'] == 2021]

    # Step 3: age group by decade
    df['age_group'] = df['age'] // 10

    # Step 4: aggregate clicks and total events per age group
    agg = df.groupby('age_group')['action'].agg(
        clicks=lambda x: (x == 'click').sum(),
        total='count'
    ).reset_index()

    # Step 5: compute CTR
    agg['ctr'] = (agg['clicks'] / agg['total']).round(4)

    # Step 6: top 3, ties broken by older age group
    result = (
        agg.sort_values(['ctr', 'age_group'], ascending=[False, False])
           .head(3)[['age_group', 'ctr']]
           .reset_index(drop=True)
    )
    return result

print(ctr_by_age(search_events, users))

   age_group     ctr
0          4  0.4000
1          3  0.4000
2          2  0.3333
